# 📓 Notebook 12 — Reflection & Self-Improvement---**Phase 5 · Advanced Deep Agent Concepts**> The best agents don't just answer — they critique their own answers and improve them.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Self-critique | Agent evaluates its own output || Retry with feedback | Loop: generate → critique → improve || Reflection loops | Structured multi-round improvement || Convergence | Know when to stop improving |### 🏗️ Build: Agent That Automatically Improves Its Answers

## 1. Setup

In [ ]:
import os, jsonfrom dotenv import load_dotenvimport litellmload_dotenv()DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")print(f"🤖 {DEFAULT_MODEL}")

---## 2. Self-Critique Pattern### The Core IdeaUse the LLM to evaluate its OWN output, then use that evaluation to generate a better version.```Generate v1 → Critique v1 → Generate v2 (using critique) → Critique v2 → ... → Done```> **Interview Insight:** This is fundamentally how RLHF works at a high level — generate, evaluate, improve. The difference is RLHF does this during training; reflection does it at inference time.

In [ ]:
def generate_and_critique(task, max_rounds=3, verbose=True):    """Generate an answer, critique it, and improve iteratively."""        # Initial generation    response = litellm.completion(        model=DEFAULT_MODEL,        messages=[{"role": "user", "content": task}],        temperature=0.7, max_tokens=500,    )    current_answer = response.choices[0].message.content        if verbose:        print(f"📝 v1: {current_answer[:150]}...")        for round_num in range(2, max_rounds + 1):        # Critique        critique_response = litellm.completion(            model=DEFAULT_MODEL,            messages=[{                "role": "system",                "content": "You are a critical reviewer. Evaluate the given answer for accuracy, completeness, clarity, and actionability. Give specific, constructive feedback. Rate quality 1-10."            }, {                "role": "user",                "content": f"Task: {task}\n\nAnswer to review:\n{current_answer}"            }],            temperature=0.3, max_tokens=300,        )        critique = critique_response.choices[0].message.content                if verbose:            print(f"\n🔍 Critique {round_num-1}: {critique[:120]}...")                # Check if quality is satisfactory (extract score)        if "9/10" in critique or "10/10" in critique:            if verbose:                print(f"  ✅ Quality sufficient — stopping at v{round_num-1}")            break                # Improve based on critique        improve_response = litellm.completion(            model=DEFAULT_MODEL,            messages=[{                "role": "user",                "content": f"Original task: {task}\n\nYour previous answer:\n{current_answer}\n\nCritique:\n{critique}\n\nWrite an improved version addressing ALL critique points."            }],            temperature=0.5, max_tokens=600,        )        current_answer = improve_response.choices[0].message.content                if verbose:            print(f"📝 v{round_num}: {current_answer[:150]}...")        return current_answer# Demoprint("🔄 Reflection Demo")print("=" * 60)result = generate_and_critique(    "Explain how a hash table works to someone preparing for a software engineering interview. Cover time complexity, collision handling, and real-world applications.",    max_rounds=3)

In [ ]:
class ReflectiveAgent:    """Agent with structured reflection and improvement loops."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        self.history = []        def reflect_and_improve(self, task, max_rounds=3, quality_threshold=8):        """Generate, reflect, and improve until quality threshold is met."""        versions = []                # Generate initial version        answer = litellm.completion(            model=self.model,            messages=[{"role": "user", "content": task}],            temperature=0.7, max_tokens=600,        ).choices[0].message.content                versions.append({"version": 1, "answer": answer, "critique": None, "score": None})        print(f"  v1 generated ({len(answer)} chars)")                for round_num in range(2, max_rounds + 1):            # Structured critique            critique_prompt = f"""Rate this answer on a scale of 1-10 and provide structured feedback.Task: {task}Answer: {answer}Respond in JSON:{{  "score": <1-10>,  "strengths": ["..."],  "weaknesses": ["..."],  "missing": ["..."],  "suggestions": ["..."]}}"""                        critique_raw = litellm.completion(                model=self.model,                messages=[{                    "role": "system",                    "content": "You are a strict quality evaluator. Be honest and specific."                }, {                    "role": "user", "content": critique_prompt                }],                response_format={"type": "json_object"},                temperature=0,            ).choices[0].message.content                        try:                critique = json.loads(critique_raw)                score = critique.get("score", 5)            except:                critique = {"score": 5, "suggestions": ["Improve overall quality"]}                score = 5                        versions[-1]["critique"] = critique            versions[-1]["score"] = score            print(f"  v{round_num-1} scored: {score}/10")                        if score >= quality_threshold:                print(f"  ✅ Quality threshold met ({score} >= {quality_threshold})")                break                        # Improve            improvement_prompt = f"""Improve this answer based on the feedback.Task: {task}Previous answer: {answer}Feedback:- Weaknesses: {critique.get('weaknesses', [])}- Missing: {critique.get('missing', [])}- Suggestions: {critique.get('suggestions', [])}Write a significantly improved version addressing ALL feedback."""                        answer = litellm.completion(                model=self.model,                messages=[{"role": "user", "content": improvement_prompt}],                temperature=0.5, max_tokens=800,            ).choices[0].message.content                        versions.append({"version": round_num, "answer": answer, "critique": None, "score": None})            print(f"  v{round_num} generated ({len(answer)} chars)")                self.history = versions        return answer        def show_evolution(self):        """Show how the answer improved across versions."""        print("\n📈 Answer Evolution:")        for v in self.history:            score = v["score"] or "pending"            print(f"  v{v['version']}: {len(v['answer'])} chars | Score: {score}")            if v["critique"] and isinstance(v["critique"], dict):                print(f"     Strengths: {v['critique'].get('strengths', [])[:2]}")                print(f"     Weaknesses: {v['critique'].get('weaknesses', [])[:2]}")agent = ReflectiveAgent()final = agent.reflect_and_improve(    "Write a production-ready Python function to validate email addresses. Include type hints, docstrings, edge cases, and tests.",    max_rounds=3, quality_threshold=8)print(f"\n📝 Final Answer:\n{final[:300]}...")agent.show_evolution()

---## 📝 Interview Quiz — Reflection & Self-Improvement### Q1: What is the reflection pattern? How does it improve agent output?<details><summary>💡 Answer</summary>Reflection is a generate → critique → improve loop:1. Agent generates initial answer2. Agent (or separate critic) evaluates the answer with specific criteria3. Agent generates improved version using critique feedback4. Repeat until quality threshold is met or max rounds reached**Why it works:** Forces the LLM to identify and fix its own weaknesses. Each round focuses on specific improvements, leading to progressively better output.</details>### Q2: How do you determine when to stop the reflection loop?<details><summary>💡 Answer</summary>Convergence criteria:1. **Quality score** crosses threshold (e.g., 8/10)2. **Diminishing returns** — score improvement < ε between rounds3. **Max rounds** — Hard limit to prevent infinite loops4. **No new critique** — Critic finds no more issues5. **Budget limit** — Token/cost budget exhausted**Risk of not stopping:** Infinite loops, wasted tokens, or *degraded* quality (over-editing can make text worse).</details>### Q3: Can reflection degrade quality? When?<details><summary>💡 Answer</summary>Yes:- **Over-editing**: The model adds unnecessary complexity trying to address every critique- **Critique drift**: The critic focuses on style over substance in later rounds- **Contradiction**: One round's improvement conflicts with a previous strength- **Hallucination amplification**: The model invents details to address "missing" feedback**Mitigation:** Use quality scoring, detect score degradation, keep the best version (not always the last).</details>### Q4: Compare reflection at inference time vs RLHF at training time.<details><summary>💡 Answer</summary>| Aspect | Inference-time Reflection | RLHF Training ||-|-|-|| When | At query time | During model training || Cost | Per-query (tokens × rounds) | One-time (GPU hours) || Flexibility | Any criteria, any round | Fixed reward model || Permanence | Per-query (no lasting learning) | Permanent improvement || Speed | Slow (multiple API calls) | No runtime overhead |**Insight:** Reflection is like "test-time compute scaling" — spending more compute at inference for better results.</details>

---## ✅ Notebook 12 Summary| Concept | Key Takeaway ||---------|-------------|| Self-critique | LLM evaluates its own output for weaknesses || Reflection loop | Generate → critique → improve → repeat || Structured critique | JSON with score, strengths, weaknesses, suggestions || Convergence | Stop when quality threshold met or diminishing returns || Risk | Over-editing can degrade quality; keep best version |### ➡️ Next: [Notebook 13 — Toolformer Style Agents](./13_toolformer.ipynb)